<a href="https://colab.research.google.com/github/yuprotsyk/bigdata-course/blob/main/notebooks/topic04_spark_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Аналіз та обробка великих даних

Ю.С. Процик. Курс лекцій

# Тема 4. Побудова масштабованих ML-систем у Spark ML

### План

1. [Вступ до Spark ML та парадигма DataFrame](#1.-Вступ-до-Spark-ML-та-парадигма-DataFrame)
2. [Основні абстракції: Transformer та Estimator](#2.-Основні-абстракції:-Transformer-та-Estimator)
3. [ML Pipeline: Оркестрація процесів](#3.-ML-Pipeline:-Оркестрація-процесів)
4. [Параметри компонентів Spark ML](#4.-Параметри-компонентів-Spark-ML)
5. [Підготовка ознак (Feature Engineering)](#5.-Підготовка-ознак-\(Feature-Engineering\))
6. [Базові алгоритми машинного навчання у Spark ML](#6.-Базові-алгоритми-машинного-навчання-у-Spark-ML)
7. [ML Persistence: Збереження та завантаження PipelineModel](#7.-ML-Persistence:-Збереження-та-завантаження-PipelineModel)
8. [Практичний end-to-end приклад для табличних даних](#8.-Практични-end\-to\-end-приклад-для-табличних-даних)
9. [Корисні ресурси](#9.-Корисні-ресурси)

## 1. Вступ до Spark ML та парадигма DataFrame

Бібліотека `spark.ml` – це сучасний стандарт Spark для машинного навчання, побудований на базі DataFrames.

**DataFrame-based API:** На відміну від застарілого `spark.mllib` (на RDD), цей API використовує Catalyst Optimizer для побудови логічних планів обчислень та Tungsten для прямого управління пам'яттю.

У Spark ML робота з алгоритмами базується на чітко визначеному "контракті" (інтерфейсі) між даними та моделлю. Це означає, що DataFrame повинен мати певну структуру стовпців, щоб алгоритм міг з ними працювати:

- **`label` (мітка):** цільова змінна, яку модель має навчитися передбачати.  
  **Важливо:** навіть для задач класифікації мітка має бути числового типу (`Double`), тому текстові категорії попередньо перетворюються на індекси.

- **`features` (ознаки):** головний вхідний стовпець. Це єдиний вектор (об'єкт типу `Vector`), що об'єднує всі незалежні змінні. Spark ML не приймає окремі стовпці як ознаки – їх обов'язково треба "зібрати" у вектор.

- **`prediction` (передбачення):** стовпець, який модель автоматично створює під час роботи. Він містить результат прогнозу.

## 2. Основні абстракції: Transformer та Estimator

Архітектура `spark.ml` базується на чіткому розмежуванні між інструментами обробки даних та алгоритмами навчання. Це реалізовано через дві ключові абстракції.

### Transformers

**Transformer** – це програмна одиниця, яка перетворює дані. Вона включає як інструменти підготовки ознак (*feature transformers*), так і вже навчені моделі.

- **Метод:** Реалізує `.transform()`.
- **Принцип:** Приймає один DataFrame і повертає новий DataFrame, зазвичай додаючи до нього нові стовпці.

**Приклади:**

- **Feature Transformer:** `Tokenizer` бере стовпець із текстом і додає стовпець із масивом слів.
- **Model:** Навчена модель `LogisticRegressionModel` бере вектори ознак і додає стовпець із прогнозованими значеннями.

### Estimators

**Estimator** – це абстракція алгоритму навчання, який аналізує вхідні дані для побудови моделі.

- **Метод:** Реалізує `.fit()`.
- **Принцип:** Приймає DataFrame і повертає об'єкт `Model` (яка, у свою чергу, стає Transformer для майбутніх даних).

**Приклад:** Алгоритм `LogisticRegression` – це Estimator. Коли ви викликаєте `.fit()` на тренувальних даних, Spark обчислює ваги та повертає готову модель `LogisticRegressionModel`.

### Властивості компонентів

#### Stateless (без збереження стану)

Методи `transform()` та `fit()` не змінюють стан вхідних об'єктів, а генерують нові структури даних.

**Чому це важливо:** Така архітектура є критичною для горизонтального масштабування. Це дозволяє Spark ефективно розпаралелювати завдання між тисячами вузлів у кластері, оскільки вузлам не потрібно синхронізувати спільний стан (*Shared State*) під час обчислень.

*В офіційній документації зазначається, що у майбутньому можуть підтримуватися stateful алгоритми через альтернативні концепції.*

#### Unique ID (унікальний ідентифікатор)

Кожен створений екземпляр Transformer чи Estimator отримує свій унікальний ID.

**Приклад:** Якщо ви будуєте складний конвеєр і використовуєте два різні об'єкти `StringIndexer` (один для стовпця **"Місто"**, інший для стовпця **"Професія"**), `Unique ID` гарантує, що Spark точно знатиме, який словник категорій належить конкретному стовпцю під час збереження або тестування моделі.

![Transformer та Estimator](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-transformer-estimator.svg)

## 3. ML Pipeline: Оркестрація процесів

У машинному навчанні зазвичай виконується послідовність алгоритмів для обробки та навчання на даних. Наприклад, стандартний робочий процес (workflow) обробки тексту включає:

- розбиття тексту документів на слова (*tokenization*)
- перетворення слів у числові вектори ознак (*feature extraction*)
- навчання прогнозної моделі на основі цих векторів та міток (*training*)

Spark ML представляє такий процес як **Pipeline (конвеєр)**, що складається з послідовності етапів (**PipelineStages**), які виконуються у строго визначеному порядку.

### Як це працює

**Pipeline** – це послідовність етапів, кожен з яких є або **Transformer**, або **Estimator**. Процес обробки даних відбувається так:

- На етапі **Transformer** викликається метод `transform()`, який на основі вхідного DataFrame створює новий і передає його як вхідні дані для наступного етапу конвеєра.
- На етапі **Estimator** спочатку викликається метод `fit()`, який створює навчену модель (Transformer). Ця модель стає частиною **PipelineModel**, а її метод `transform()` застосовується до DataFrame для продовження ланцюжка обробки.


### Ілюстрація Pipeline на прикладі текстових документів

#### Навчання конвеєра

Розглянемо простий workflow для текстових документів під час навчання:

1. Розбити текст документа на слова.  
2. Перетворити слова в числовий вектор ознак.  
3. Навчити модель на основі векторів ознак та міток.

![ML Pipeline](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-mlPipeline.png)

Наведена вище схема демонструє **Pipeline з трьома етапами**:

- **Перші два етапи** (Tokenizer і HashingTF) – це **Transformers** (сині).  
- **Третій етап** (LogisticRegression) – це **Estimator** (червоний).  

Нижній рядок відображає потік даних через Pipeline, де **циліндри** символізують **DataFrames**.

**Пояснення етапів:**

1. Викликається `Pipeline.fit()` на початковому DataFrame з текстовими документами та мітками.  
2. `Tokenizer.transform()` розбиває текст на слова, додаючи новий стовпець зі словами у DataFrame.  
3. `HashingTF.transform()` перетворює стовпець зі словами на вектори ознак, додаючи новий стовпець у DataFrame.  
4. Оскільки `LogisticRegression` – це Estimator, Pipeline викликає `LogisticRegression.fit()`, щоб створити модель `LogisticRegressionModel`.  
5. Якби Pipeline мав більше Estimators, він викликав би `transform()` навченої моделі на DataFrame перед передачею його наступному етапу.

> **Важливо:** **Pipeline** сам є **Estimator**.  
> Після виклику `fit()` він повертає **PipelineModel**, яка є **Transformer** у якому всі Estimators (наприклад, алгоритм логістичної регресії) вже перетворені на навчені моделі. Це гарантує, що тестові дані пройдуть через ідентичні кроки обробки ознак, що і навчальні.

#### Використання PipelineModel на тестових даних

![ML Pipeline Test](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-mlPipelineModel.png)

### Проблема витоку даних (Data Leakage)

**Pipeline** – це **найкращий спосіб уникнути витоку даних**.

Обчислення статистичних показників (наприклад, середнього значення для нормалізації) має відбуватися виключно через `Pipeline.fit()` на тренувальній вибірці. Якщо розрахувати ці параметри на всьому датасеті до його розбиття, інформація з тестових даних потрапить в процес навчання. Це призведе до **перенавчання (overfitting)**: модель продемонструє аномально високу точність на тестах, але виявиться недієздатною на реальних нових даних.

### Архітектура на основі графа (DAG)

Навіть якщо конвеєр виглядає як пряма послідовність кроків, Spark ML завжди розглядає його як **напрямлений ациклічний граф (DAG)**. Це означає, що потік даних керується не просто порядком у списку, а логічними зв'язками між етапами.

#### Як це працює для всіх Pipeline

Зв’язки між етапами визначаються **неявно через назви стовпців**. Наприклад, якщо Етап Б потребує на вхід стовпець `words`, він автоматично стає залежним від Етапу А, який цей стовпець створює. Назви стовпців – це своєрідні "адреси", за якими дані передаються всередині графа.

#### Критична вимога – топологічний порядок

Оскільки це граф, етапи в масиві `stages` мають бути вказані **в строго топологічному порядку** (щоб дані для стовпця були створені раніше, ніж їх захоче використати наступний алгоритм).

#### Можливість нелінійності

Розуміння Pipeline як графа дозволяє за потреби будувати **складні структури**. Наприклад, коли один вхідний стовпець розгалужується на кілька паралельних гілок обробки, результати яких потім знову об'єднуються перед навчанням моделі.

### Перевірка типів під час виконання (Runtime Checking)

Для DataFrame Spark не може перевірити типи даних на етапі компіляції.

**Рішення:**  
Перед запуском обчислень Pipeline проводить інспекцію схеми. Якщо Estimator очікує вектор у стовпці `features`, а отримує рядок (`String`), Spark видасть помилку негайно, не чекаючи години обробки даних у кластері.

### Унікальність екземплярів

Важливо, щоб кожен етап у конвеєрі був **унікальним об'єктом**.

**Чому:**  
Кожен етап повинен мати власний **Unique ID**. Не можна вставити один і той самий об'єкт `myHashingTF` у конвеєр двічі. Якщо потрібно виконати подібну дію двічі, слід створити **два окремі екземпляри** об'єкта.

![ML Pipeline](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-ML-pipeline.svg)

### Практична реалізація: Побудова обчислювального конвеєра в PySpark

Розглянемо приклад створення конвеєра для класифікації текстових повідомлень.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkML") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import HashingTF, Tokenizer

# 1. Створення вхідних даних
# Містить стовпці: id, text (сирі дані) та label (цільова змінна)
training = spark.createDataFrame([
    (0, "spark machine learning is powerful", 1.0),
    (1, "bad performance and errors", 0.0),
    (2, "spark ml pipelines are easy", 1.0),
    (3, "hadoop mapreduce outdated", 0.0)
], ["id", "text", "label"])

# 2. Визначення етапів (Stages)
# Tokenizer: розбиває текст на масив слів
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# HashingTF: перетворює слова у вектор ознак (features)
hashingTF = HashingTF(inputCol="words", outputCol="features")

# LogisticRegression: Estimator, що шукає залежність між features та label
lr = LogisticRegression(maxIter=10, regParam=0.001)

# 3. Складання конвеєра
# Порядок у списку визначає топологічний порядок виконання
pipeline = Pipeline(stages=[tokenizer, hashingTF, lr])

# 4. Навчання (Pipeline.fit() повертає PipelineModel)
# У цей момент виконуються методи .transform() для токенізатора/хешування
# та метод .fit() для LogisticRegression
model = pipeline.fit(training)

# Тепер об'єкт 'model' готовий до використання на тестових даних

# Застосуємо навчену модель до тих самих даних (для демонстрації)
predictions = model.transform(training)

# Подивимося, які нові стовпці з'явилися
predictions.printSchema()

# Подивимося на реальний вміст (як текст став вектором і прогнозом)
predictions.select("text", "words", "features", "prediction").show(truncate=False)

root
 |-- id: long (nullable = true)
 |-- text: string (nullable = true)
 |-- label: double (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)

+----------------------------------+----------------------------------------+-----------------------------------------------------------------+----------+
|text                              |words                                   |features                                                         |prediction|
+----------------------------------+----------------------------------------+-----------------------------------------------------------------+----------+
|spark machine learning is powerful|[spark, machine, learning, is, powerful]|(262144,[9144,106841,163984,173558,251904],[1.0,1.0,1.0,1.0,1.0])|1.0       |
|bad performan

## 4. Параметри компонентів Spark ML

Компоненти ML-конвеєрів (**Estimators** та **Transformers**) використовують **уніфікований API** для визначення та налаштування параметрів.

### Основні поняття

- **`Param`** – це іменований параметр, який містить вбудовану документацію.
- **`ParamMap`** – це набір пар **"параметр-значення"** (*set of parameter-value pairs*).

### Два способи передачі параметрів

- **Встановлення для конкретного екземпляра:**  
  Можна викликати методи-сетери безпосередньо у об’єкта (наприклад, `lr.setMaxIter(10)`), щоб алгоритм використовував ці налаштування під час навчання.

- **Передача через `ParamMap`:**  
  Можна передати об'єкт `ParamMap` безпосередньо у методи `.fit()` або `.transform()`.

### Пріоритетність та перевизначення

Будь-які параметри, передані через `ParamMap`, мають **вищий пріоритет** і **перезаписують** (*override*) параметри, які були встановлені раніше за допомогою методів-сетерів.

### Унікальність та ідентифікація

Параметри належать до **конкретних екземплярів компонентів**. Оскільки кожен екземпляр Transformer чи Estimator має свій **унікальний ID**, це дозволяє гнучко налаштовувати конвеєр.

Наприклад, якщо у вашому `Pipeline` є два об'єкти `LogisticRegression` (`lr1` та `lr2`), ви можете створити один `ParamMap`, де вкажете різні значення максимальної кількості ітерацій для кожного з них окремо.

### Інструменти перевірки

- **`explainParams()`** – цей метод дозволяє отримати список усіх доступних параметрів об'єкта разом із їхніми описами та значеннями за замовчуванням.
- **`extractParamMap()`** – метод дозволяє переглянути параметри, які фактично використовувалися під час навчання моделі.

Така архітектура дозволяє використовувати **один і той самий екземпляр алгоритму** з різними наборами параметрів у різних частинах коду, не змінюючи сам об'єкт.

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.linalg import Vectors

# Підготовка тренувальних даних (мітка та вектори ознак)
training = spark.createDataFrame([
    (1.0, Vectors.dense([0.0, 1.1, 0.1])),
    (0.0, Vectors.dense([2.0, 1.0, -1.0])),
    (0.0, Vectors.dense([2.0, 1.3, 1.0])),
    (1.0, Vectors.dense([0.0, 1.2, -0.5]))
], ["label", "features"])

# Створення екземпляра LogisticRegression. Це Estimator
lr = LogisticRegression()

# Перегляд усіх параметрів, їхніх описів та значень за замовчуванням
print("Доступні параметри та їх опис:")
print(lr.explainParams())

# Шлях №1: Встановлення параметрів через сетери
lr.setMaxIter(10).setRegParam(0.01)

# Навчання першої моделі з використанням параметрів, встановлених через сетери
model1 = lr.fit(training)
print("\nПараметри Model 1 (встановлені через сетери):")
print(model1.extractParamMap())

# Шлях №2: Використання ParamMap для перевизначення
# Ми створюємо словник, де ключами є об'єкти параметрів самого екземпляра lr
paramMap = {lr.maxIter: 20}
paramMap[lr.maxIter] = 30  # Перезаписуємо значення
paramMap.update({lr.regParam: 0.1, lr.threshold: 0.55}) # Додаємо кілька параметрів

# Також можна змінити назву вихідного стовпця
paramMap2 = {lr.probabilityCol: "myProbability"}
paramMapCombined = paramMap.copy()
paramMapCombined.update(paramMap2)

# Навчання другої моделі з ParamMap
# paramMapCombined ПЕРЕЗАПИСУЄ всі параметри, встановлені раніше через сетери
model2 = lr.fit(training, paramMapCombined)

print("\nПараметри Model 2 (перезаписані через ParamMap):")
print(model2.extractParamMap())

# Застосування моделі
# Тепер model2 буде виводити ймовірності у стовпець "myProbability", а не "probability"
test_data = spark.createDataFrame([(1.0, Vectors.dense([-1.0, 1.5, 1.3]))], ["label", "features"])
prediction = model2.transform(test_data)
prediction.select("features", "myProbability", "prediction").show()


Доступні параметри та їх опис:
aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0)
family: The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial (default: auto)
featuresCol: features column name. (default: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label)
lowerBoundsOnCoefficients: The lower bounds on coefficients if fitting under bound constrained optimization. The bound matrix must be compatible with the shape (1, number of features) for binomial regression, or (number of classes, number of features) for multinomial regression. (undefined)
lowerBoundsOnIntercepts: The lower bounds on intercepts if fitting under bound constrained optimization. The bou

## 5. Підготовка ознак (Feature Engineering)

Перш ніж передати дані будь-якому алгоритму Spark ML, їх потрібно привести до єдиного формату – стовпця `features` типу `Vector`. Spark ML ділить весь інструментарій на три групи:

| Група | Призначення |
|-------|-------------|
| **Extraction** | Вилучення ознак із "сирих" даних |
| **Transformation** | Масштабування, кодування, перетворення наявних ознак |
| **Selection** | Відбір найбільш інформативної підмножини ознак |

> [Офіційна документація: Extracting, transforming and selecting features](https://spark.apache.org/docs/latest/ml-features.html)

### Dense vs Sparse Vector

Стовпець `features` зберігає об'єкти типу `Vector`. Існує два представлення:

**Dense Vector** – зберігає всі значення підряд, включно з нулями:
`[1.0, 0.0, 0.0, 0.0, 0.0, 3.0]`

**Sparse Vector** – зберігає лише розмір, індекси та ненульові значення:
`(6, [0, 5], [1.0, 3.0])`

Обидва представлення **математично еквівалентні** – алгоритми працюють з ними однаково. Різниця лише у витратах пам'яті.

Sparse є критично важливим для NLP: якщо словник містить 100 000 слів, але в документі є лише 50 – Dense Vector витратить пам'ять на 99 950 нулів.

| Компонент (Stage) | Формат виходу |
|-------------|---------------|
| `VectorAssembler` (числові ознаки) | Dense |
| `HashingTF`, `CountVectorizer`, `OneHotEncoder` | Sparse |
| `MaxAbsScaler` (зі Sparse входом) | Sparse (зберігає розрідженість) |

In [ ]:
from pyspark.ml.linalg import Vectors

# Dense Vector – зберігає всі 6 значень
dense_vec = Vectors.dense([1.0, 0.0, 0.0, 0.0, 0.0, 3.0])

# Sparse Vector – зберігає лише розмір + ненульові елементи
# Vectors.sparse(розмір, [індекси], [значення])
sparse_vec = Vectors.sparse(6, [0, 5], [1.0, 3.0])

print(f"Dense:  {dense_vec}")
print(f"Sparse: {sparse_vec}")
print(f"Еквівалентні: {dense_vec == sparse_vec}")  # True

Dense:  [1.0,0.0,0.0,0.0,0.0,3.0]
Sparse: (6,[0,5],[1.0,3.0])
Еквівалентні: True


### Карта інструментів

Основні класи Spark ML для роботи з ознаками:
![ML ETS](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-ETS.svg)

#### Extraction

| Клас | Тип | Коли використовувати |
|------|-----|----------------------|
| `HashingTF` + `IDF` | Transformer + Estimator | TF-IDF для тексту, великі корпуси, без потреби у словнику |
| `CountVectorizer` + `IDF` | Estimator + Estimator | TF-IDF коли потрібен явний словник (напр. для LDA) |
| `Word2Vec` | Estimator | Семантичні embedding-вектори слів |
| `FeatureHasher` | Transformer | Багато різнотипних стовпців → один вектор, швидке прототипування |


#### Transformation

| Клас | Тип | Коли використовувати |
|------|-----|----------------------|
| `Tokenizer` / `RegexTokenizer` | Transformer | Розбиття тексту на токени |
| `StopWordsRemover` | Transformer | Видалення незначущих слів |
| `NGram` | Transformer | Генерація n-грам |
| `StringIndexer` → `OneHotEncoder` | Estimator → Estimator | Кодування категоріальних ознак |
| `TargetEncoder` | Estimator | Альтернатива OHE для дерев (Spark 4.0+) |
| `StandardScaler` | Estimator | Числові ознаки, лінійні моделі |
| `MinMaxScaler` | Estimator | Потрібен фіксований діапазон [0,1] |
| `MaxAbsScaler` | Estimator | Розріджені дані – не зміщує нулі |
| `RobustScaler` | Estimator | Дані з викидами (Spark 3.0+) |
| `Binarizer` / `Bucketizer` / `QuantileDiscretizer` | Transformer / Transformer / Estimator | Дискретизація числових ознак |
| `VectorAssembler` | Transformer | Обов'язковий фінальний крок – збирає всі ознаки у вектор |
| `Imputer` | Estimator | Заповнення пропущених значень |
| `PCA` | Estimator | Зниження розмірності |
| `SQLTransformer` | Transformer | Перетворення через SQL-вирази |

#### Selection

| Клас | Тип | Коли використовувати |
|------|-----|----------------------|
| `VectorSlicer` | Transformer | Відбір ознак за відомими індексами |
| `ChiSqSelector` | Estimator | Категоріальні ознаки, задача класифікації |
| `UnivariateFeatureSelector` | Estimator | Гнучкіший відбір: різні типи ознак і міток (Spark 3.1+) |
| `VarianceThresholdSelector` | Estimator | Відкидання майже константних ознак (Spark 3.1+) |
| `RFormula` | Estimator | R-подібний синтаксис, швидке прототипування |

### Extraction

#### TF-IDF

TF-IDF вимірює важливість терміна для документа відносно всього корпусу. Слова, що є в кожному документі ("і", "або", "the"), отримують низьку вагу.

$$IDF(t, D) = \log \frac{|D| + 1}{DF(t, D) + 1}, \quad TFIDF(t,d,D) = TF(t,d) \cdot IDF(t,D)$$

У Spark ML TF і IDF розділені свідомо – для гнучкості:

- **`HashingTF`** (Transformer) – швидко, без словника, можливі колізії хешів, розмір за замовчуванням $2^{18}$.
- **`CountVectorizer`** (Estimator) – будує явний словник, підтримує `minDF` та `maxDF`, дозволяє зворотне відображення.
- **`IDF`** (Estimator) – застосовується поверх будь-якого з двох вище.

#### Word2Vec

Навчає нейромережу і відображає кожне слово на щільний вектор фіксованої розмірності. Слова зі схожим контекстом – близькі у векторному просторі. На виході `transform()` повертає середній вектор усіх слів документа.

#### FeatureHasher

Проектує набір різнотипних стовпців (числові, рядкові, булеві) у єдиний Sparse Vector через hashing trick. Не потребує `fit()`. Рядкові ознаки кодуються як `"назва_стовпця=значення"` з вагою 1.0, числові зберігають своє значення як вагу.

In [ ]:
from pyspark.ml.feature import Tokenizer, HashingTF, IDF

sentenceData = spark.createDataFrame([
    (0.0, "Hi I heard about Spark"),
    (0.0, "I wish Java could use case classes"),
    (1.0, "Logistic regression models are neat")
], ["label", "sentence"])

tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
wordsData  = tokenizer.transform(sentenceData)

# HashingTF: numFeatures=20 лише для демонстрації, за замовч. 2^18
hashingTF    = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20)
featurized   = hashingTF.transform(wordsData)

idfModel     = IDF(inputCol="rawFeatures", outputCol="features").fit(featurized)
rescaled     = idfModel.transform(featurized)

# Слова присутні в усіх 3 документах отримають IDF=0
rescaled.select("label", "features").show(truncate=False)

+-----+-------------------------------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                                                   |
+-----+-------------------------------------------------------------------------------------------------------------------------------------------+
|0.0  |(20,[6,8,13,16],[0.28768207245178085,0.6931471805599453,0.28768207245178085,0.5753641449035617])                                           |
|0.0  |(20,[0,2,7,13,15,16],[0.6931471805599453,0.6931471805599453,1.3862943611198906,0.28768207245178085,0.6931471805599453,0.28768207245178085])|
|1.0  |(20,[3,4,6,11,19],[0.6931471805599453,0.6931471805599453,0.28768207245178085,0.6931471805599453,0.6931471805599453])                       |
+-----+---------------------------------------------------------------------------------------------------------

### Transformation – Перетворення ознак

#### Текстові Transformers

Типовий NLP-pipeline у Spark ML:

![NLP-pipeline](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-NLP-pipeline.svg)

- **`Tokenizer`** – розбиває рядок за пробілами, приводить до нижнього регістру.
- **`RegexTokenizer`** – те саме, але за довільним regex-патерном. Додатковий параметр `minTokenLength` відкидає короткі токени.
- **`StopWordsRemover`** – видаляє стоп-слова. Вбудовані списки для багатьох мов. Spark 3.0+: підтримує `inputCols`/`outputCols`.
- **`NGram`** – перетворює масив токенів на послідовність n-грам. При `n=2`: `["Hi", "I", "heard"]` → `["Hi I", "I heard"]`.

#### Кодування категоріальних ознак

Алгоритми Spark ML приймають лише числові значення. Стандартний двокроковий процес для категоріальних ознак:

![StringIndexer OHE](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img04-StringIndexer-OHE.svg)

- **`StringIndexer`** (Estimator) – рядок → числовий індекс. Порядок: найчастіша категорія = 0. Параметр `handleInvalid` визначає поведінку при нових категоріях під час `transform()`: `"error"` / `"skip"` / `"keep"`.
- **`OneHotEncoder`** (Estimator) – індекс → бінарний Sparse Vector розмірності $N-1$. Остання категорія відкидається (`dropLast=True`) для уникнення мультиколінеарності.
- **`IndexToString`** (Transformer) – зворотне перетворення після прогнозу моделі: індекс → вихідний рядок.
- **`TargetEncoder`** (Estimator, Spark 4.0+) – альтернатива OHE для алгоритмів на основі дерев: замінює категорію на середнє значення цільової змінної по цій категорії.

> Spark 3.0+: `StringIndexer` та `OneHotEncoder` підтримують `inputCols`/`outputCols` – обробка кількох стовпців одним `fit()`.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, IndexToString

df = spark.createDataFrame([
    (0, "cat"), (1, "dog"), (2, "cat"),
    (3, "bird"), (4, "dog"), (5, "cat")
], ["id", "animal"])

# StringIndexer: cat(3x)→0, dog(2x)→1, bird(1x)→2
indexer       = StringIndexer(inputCol="animal", outputCol="animalIndex",
                               handleInvalid="keep")
indexer_model = indexer.fit(df)
indexed       = indexer_model.transform(df)
print("Словник (за спаданням частоти):", indexer_model.labels)

# OneHotEncoder: 3 категорії → вектор розмірності 2 (N-1)
encoder = OneHotEncoder(inputCols=["animalIndex"], outputCols=["animalVec"])
encoded = encoder.fit(indexed).transform(indexed)
encoded.select("animal", "animalIndex", "animalVec").show()

# IndexToString: відновлення рядків після прогнозу
converter = IndexToString(inputCol="animalIndex", outputCol="originalAnimal",
                          labels=indexer_model.labels)
converter.transform(indexed).select("animalIndex", "originalAnimal").show()

Словник (за спаданням частоти): ['cat', 'dog', 'bird']
+------+-----------+-------------+
|animal|animalIndex|    animalVec|
+------+-----------+-------------+
|   cat|        0.0|(3,[0],[1.0])|
|   dog|        1.0|(3,[1],[1.0])|
|   cat|        0.0|(3,[0],[1.0])|
|  bird|        2.0|(3,[2],[1.0])|
|   dog|        1.0|(3,[1],[1.0])|
|   cat|        0.0|(3,[0],[1.0])|
+------+-----------+-------------+

+-----------+--------------+
|animalIndex|originalAnimal|
+-----------+--------------+
|        0.0|           cat|
|        1.0|           dog|
|        0.0|           cat|
|        2.0|          bird|
|        1.0|           dog|
|        0.0|           cat|
+-----------+--------------+



#### Масштабування числових ознак

Алгоритми, що базуються на відстанях (k-means, SVM, KNN) критично залежать від масштабу ознак. Усі scalers є **Estimators** – вони навчаються на тренувальних даних і зберігають статистики в моделі.

| Клас | Формула        | Коли обирати |
|------|----------------|--------------|
| `StandardScaler` | $\frac{x-\mu}{\sigma}$ | Ознаки мають різні одиниці вимірювання, а алгоритм чутливий до масштабу (PCA, Logistic Regression) |
| `MinMaxScaler` | $\frac{x-x_{\min}}{x_{\max}-x_{\min}}$ | Потрібен фіксований діапазон, напр. для нейронних мереж. Перетворює Sparse → Dense |
| `MaxAbsScaler` | $\frac{x}{|x_{\max}|}$ | Розріджені дані — зберігає нулі, не зміщує |
| `RobustScaler` | $\frac{x-\mathrm{median}}{\mathrm{IQR}}$ | Дані з викидами (Spark 3.0+) |

#### Дискретизація числових ознак

Іноді числову ознаку доцільно перетворити на категоріальну:

- **`Binarizer`** (Transformer) – число → 0 або 1 за заданим порогом.
- **`Bucketizer`** (Transformer) – число → дискретний бакет за межами, заданими вручну.
- **`QuantileDiscretizer`** (Estimator) – те саме, але межі бакетів визначаються автоматично за квантилями даних.

---

#### VectorAssembler – обов'язковий фінальний крок

`VectorAssembler` (Transformer) об'єднує кілька числових стовпців та/або векторних стовпців в **один стовпець `features`** типу `Vector`. Без нього жоден алгоритм Spark ML не прийме дані.

Приймає числові типи та тип `vector`. Параметр `handleInvalid`: `"error"` / `"skip"` / `"keep"` для обробки `null`-значень.

---

#### Imputer – заповнення пропущених значень

`Imputer` (Estimator) заповнює `null` у числових стовпцях через `mean`, `median` або `mode`. Підтримує `inputCols`/`outputCols`. Важливо: **не підтримує категоріальні ознаки**.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, Imputer

df = spark.createDataFrame([
    (25.0, 50000.0, 1.0),
    (35.0, 80000.0, 0.0),
    (None, 60000.0, 1.0),   # пропущене значення
    (45.0, None,   0.0),    # пропущене значення
], ["age", "salary", "label"])

# Крок 1: Imputer – заповнюємо пропущені значення медіаною
imputer = Imputer(inputCols=["age", "salary"],
                  outputCols=["age_imp", "salary_imp"],
                  strategy="median")
imputed = imputer.fit(df).transform(df)

# Крок 2: VectorAssembler – збираємо всі числові ознаки у вектор
assembler = VectorAssembler(inputCols=["age_imp", "salary_imp"],
                             outputCol="raw_features")
assembled = assembler.transform(imputed)

# Крок 3: StandardScaler – стандартизуємо вектор ознак
scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                        withMean=True, withStd=True)
scaled = scaler.fit(assembled).transform(assembled)

scaled.select("age", "salary", "features").show(truncate=False)

+----+-------+----------------------------------------+
|age |salary |features                                |
+----+-------+----------------------------------------+
|25.0|50000.0|[-1.224744871391589,-0.9933992677987828]|
|35.0|80000.0|[0.0,1.390758974918296]                 |
|NULL|60000.0|[0.0,-0.1986798535597566]               |
|45.0|NULL   |[1.224744871391589,-0.1986798535597566] |
+----+-------+----------------------------------------+



### Selection – Відбір ознак

Після побудови вектора ознак він може містити сотні або тисячі вимірів. Частина з них – шум або константи, які лише заважають навчанню. Selection дозволяє автоматично відібрати найінформативнішу підмножину.

| Клас | Метод відбору | Коли використовувати |
|------|---------------|----------------------|
| `VectorSlicer` | За індексами | Коли вже відомо які саме ознаки потрібні |
| `RFormula` | R-подібний синтаксис | Швидке прототипування, звична R/statsmodels нотація |
| `ChiSqSelector` | Критерій χ² | Категоріальні ознаки + задача класифікації |
| `UnivariateFeatureSelector` | Статистичні тести | Гнучкіший вибір: підтримує різні комбінації типів ознак і міток (Spark 3.1+) |
| `VarianceThresholdSelector` | Порогова дисперсія | Швидке відкидання майже константних ознак (Spark 3.1+) |

**`VectorSlicer`** – Transformer. Приймає вектор і повертає новий вектор із вибраними індексами або іменами. Не потребує `fit()`.

**`ChiSqSelector`** – Estimator. Обчислює статистику χ² між кожною ознакою і міткою, відбирає `numTopFeatures` найзначущіших. Працює лише з невід'ємними значеннями ознак і дискретними мітками.

**`UnivariateFeatureSelector`** – Estimator (Spark 3.1+). Узагальнює `ChiSqSelector`: автоматично обирає статистичний тест залежно від типів ознак (`continuous`/`categorical`) і мітки (`continuous`/`categorical`). Підтримує різні режими відбору: `numTopFeatures`, `percentile`, `fpr`, `fdr`, `fwe`.

**`VarianceThresholdSelector`** – Estimator (Spark 3.1+). Видаляє ознаки, дисперсія яких нижча за поріг `varianceThreshold`. За замовчуванням `0.0` – відкидає лише абсолютно константні ознаки.

> [Офіційна документація: Feature Selectors](https://spark.apache.org/docs/latest/ml-features.html#feature-selectors)

## 6. Базові алгоритми машинного навчання у Spark ML

Усі алгоритми Spark ML мають **єдиний API**: є Estimators, після `fit()` повертають `Model` (Transformer). Це дозволяє вставляти будь-який алгоритм у `Pipeline` без змін у коді навколо нього.

**Вхідні та вихідні стовпці** – спільні для всіх алгоритмів з учителем:

| Стовпець | Тип | Призначення |
|----------|-----|-------------|
| `features` | `Vector` | Вхідні ознаки (обов'язково зібрані через `VectorAssembler`) |
| `label` | `Double` | Цільова змінна (навіть для класифікації – числовий індекс) |
| `rawPrediction` | `Vector` | "Сирі" логіти або голоси дерев до перетворення |
| `probability` | `Vector` | Ймовірності класів (де підтримується) |
| `prediction` | `Double` | Фінальний прогноз |

Усі назви стовпців є **значеннями за замовчуванням** і змінюються через параметри `featuresCol`, `labelCol`, `predictionCol` тощо.

**Таблиця алгоритмів:**

| Задача | Алгоритм | Клас |
|--------|----------|------|
| **Класифікація** | Logistic Regression | `LogisticRegression` |
| | Decision Tree | `DecisionTreeClassifier` |
| | Random Forest | `RandomForestClassifier` |
| | Gradient Boosted Trees | `GBTClassifier` |
| | Linear SVC | `LinearSVC` |
| | Naive Bayes | `NaiveBayes` |
| | Multilayer Perceptron | `MultilayerPerceptronClassifier` |
| | Factorization Machines | `FMClassifier` |
| **Регресія** | Linear Regression | `LinearRegression` |
| | Generalized Linear Model | `GeneralizedLinearRegression` |
| | Decision Tree Regressor | `DecisionTreeRegressor` |
| | Random Forest Regressor | `RandomForestRegressor` |
| | GBT Regressor | `GBTRegressor` |
| | Isotonic Regression | `IsotonicRegression` |
| | Factorization Machines | `FMRegressor` |
| **Кластеризація** | K-Means | `KMeans` |
| | Bisecting K-Means | `BisectingKMeans` |
| | Gaussian Mixture | `GaussianMixture` |
| | LDA | `LDA` |

> [Офіційна документація: Classification and regression](https://spark.apache.org/docs/latest/ml-classification-regression.html)

> [Офіційна документація: Clustering](https://spark.apache.org/docs/latest/ml-clustering.html)

### Класифікація

#### Logistic Regression

Базовий алгоритм бінарної та мультикласової класифікації. Spark ML підтримує три режими через параметр `family`: `auto` (за замовч.), `binomial` (бінарна), `multinomial` (мультикласова через softmax).

Підтримує регуляризацію через `elasticNetParam`:
- `0.0` → L2 (Ridge)
- `1.0` → L1 (Lasso)
- `(0, 1)` → ElasticNet

Після `fit()` модель надає `summary` з метриками на тренувальних даних: `accuracy`, `areaUnderROC`, `precisionByLabel` тощо.

---

#### Decision Tree

Інтерпретований алгоритм – результат можна пояснити як набір правил. Не потребує масштабування ознак. Після навчання доступна `featureImportances` – вектор важливості кожної ознаки.

Ключові параметри: `maxDepth` (контролює overfitting), `impurity` (`gini` або `entropy` для класифікації, `variance` для регресії), `maxBins` (≥ кількості категорій у будь-якій категоріальній ознаці).

#### Random Forest vs Gradient Boosted Trees

Обидва є ансамблями дерев – найпотужніші алгоритми загального призначення у Spark ML.

| | Random Forest | GBT |
|-|---------------|-----|
| Ідея | Багато незалежних дерев, голосування | Кожне дерево виправляє помилки попереднього |
| Навчання | Паралельне | Послідовне |
| Ризик overfitting | Нижчий | Вищий (контролюється `stepSize`) |
| Підтримка мультикласу | Так | Ні (лише бінарна класифікація) |
| `weightCol` | Spark 3.0+ | Spark 3.0+ |

> `GBTClassifier` підтримує **лише бінарну класифікацію**. Для мультикласової задачі – `RandomForestClassifier` або `LogisticRegression` з `family="multinomial"`.

#### Інші класифікатори

- **`LinearSVC`** – лінійний SVM для бінарної класифікації. Ефективний на великих розріджених даних (текст). Не виводить `probability`.
- **`NaiveBayes`** – ймовірнісний класифікатор. Типи: `multinomial` (підрахунки слів), `bernoulli` (бінарні ознаки), `gaussian` (неперервні ознаки), `complement` (незбалансовані класи, Spark 3.0+). Усі ознаки мають бути невід'ємними для `multinomial`/`bernoulli`.
- **`MultilayerPerceptronClassifier`** – нейронна мережа прямого поширення. Архітектура задається параметром `layers`: наприклад `[4, 5, 4, 3]` – 4 вхідні нейрони, два приховані шари (5 і 4), 3 вихідні класи.
- **`FMClassifier`** – Factorization Machines (Spark 3.0+). Ефективно моделює взаємодії між ознаками навіть при розріджених даних.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.linalg import Vectors

data = spark.createDataFrame([
    (1.0, Vectors.dense([0.0, 1.1, 0.1])),
    (0.0, Vectors.dense([2.0, 1.0, -1.0])),
    (0.0, Vectors.dense([2.0, 1.3,  1.0])),
    (1.0, Vectors.dense([0.0, 1.2, -0.5])),
    (1.0, Vectors.dense([0.1, 1.5,  0.2])),
    (0.0, Vectors.dense([1.8, 0.9,  1.5])),
], ["label", "features"])

train, test = data.randomSplit([0.8, 0.2], seed=42)

# Logistic Regression: L2 регуляризація
lr = LogisticRegression(maxIter=20, regParam=0.01, elasticNetParam=0.0)
lr_model = lr.fit(train)

print(f"Коефіцієнти: {lr_model.coefficients}")
print(f"AUC на train: {lr_model.summary.areaUnderROC:.4f}")
lr_model.transform(test).select("label", "probability", "prediction").show()

# Random Forest: ансамбль з 20 дерев
rf = RandomForestClassifier(numTrees=20, maxDepth=4, seed=42)
rf_model = rf.fit(train)

print(f"Важливість ознак: {rf_model.featureImportances}")
rf_model.transform(test).select("label", "prediction").show()

Коефіцієнти: [-2.7838538963617596,5.585816485557255,-0.789064219433307]
AUC на train: 1.0000
+-----+--------------------+----------+
|label|         probability|prediction|
+-----+--------------------+----------+
|  1.0|[0.12814966246930...|       1.0|
|  0.0|[0.99510858998733...|       0.0|
+-----+--------------------+----------+

Важливість ознак: (3,[0,1,2],[0.7333333333333333,0.0625,0.2041666666666667])
+-----+----------+
|label|prediction|
+-----+----------+
|  1.0|       1.0|
|  0.0|       0.0|
+-----+----------+



### Регресія

#### Linear Regression

Базовий алгоритм для прогнозування неперервних значень. Підтримує дві функції втрат:
- `squaredError` – стандартний МНК (метод найменших квадратів); ефективний на "чистих" даних, але чутливий до викидів (за замовчуванням).
- `huber` – гібридна функція: квадратична для малих помилок і абсолютна для великих. Стійка до викидів.

Регуляризація через `elasticNetParam` – аналогічно до `LogisticRegression`: `0.0` = L2, `1.0` = L1.

Після `fit()` модель надає `summary` з метриками на тренувальних даних: `RMSE`, `r2`, `MAE`, `explainedVariance`.

#### Generalized Linear Model (GLM)

`GeneralizedLinearRegression` – узагальнення лінійної регресії для випадків, коли цільова змінна не обов’язково має нормальний розподіл. Модель гнучко налаштовується через два ключові параметри:

- `family`**(розподіл):** визначає природу даних (неперервні, цілі числа, ймовірності).
- `link` **(функція зв’язку):** математичне перетворення, що пов'язує лінійну модель із цільовою змінною, гарантуючи, що прогноз не вийде за логічні межі (наприклад, не буде від'ємним для підрахунку подій).

Підтримувані сімейства розподілів:

| `family` | Типова задача | `link` за замовчуванням |
|----------|---------------|-------------------|
| `gaussian` | Неперервна цільова змінна | `identity` |
| `binomial` | Бінарна класифікація | `logit` |
| `poisson` | Підрахунок подій (цілі числа ≥ 0) | `log` |
| `gamma` | Додатні неперервні значення, права асиметрія | `inverse` |
| `tweedie` | Змішані нулі та додатні значення | `power` |

> `GeneralizedLinearRegression` підтримує максимум **4096 ознак**. Для більшої кількості – `LinearRegression`.

#### Ансамблеві методи для регресії

`DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor` – ті самі алгоритми що і для класифікації, але для неперервної цільової змінної. Критерій розщеплення – `variance` (мінімізація дисперсії). Прогноз – середнє значень у листку.

`GBTRegressor` підтримує різні функції втрат через параметр `lossType`: `squared` (за замовч.) та `absolute` (стійка до викидів).

> [Офіційна документація: Classification and regression](https://spark.apache.org/docs/latest/ml-classification-regression.html)

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.linalg import Vectors

data = spark.createDataFrame([
    (1.0,  Vectors.dense([1.0, 0.5])),
    (2.5,  Vectors.dense([2.0, 1.0])),
    (3.8,  Vectors.dense([3.0, 1.5])),
    (5.1,  Vectors.dense([4.0, 2.0])),
    (6.3,  Vectors.dense([5.0, 2.5])),
    (7.9,  Vectors.dense([6.0, 3.0])),
    (9.0,  Vectors.dense([7.0, 3.5])),
    (10.2, Vectors.dense([8.0, 4.0])),
], ["label", "features"])

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0,   # L2
    loss="squaredError"
)
lr_model = lr.fit(train)

# Коефіцієнти навченої моделі
print(f"Коефіцієнти: {lr_model.coefficients}")
print(f"Intercept:   {lr_model.intercept:.4f}")

# Summary на тренувальних даних
summary = lr_model.summary
print(f"RMSE: {summary.rootMeanSquaredError:.4f}")
print(f"R²:   {summary.r2:.4f}")
print(f"MAE:  {summary.meanAbsoluteError:.4f}")

# Прогноз на тестових даних
lr_model.transform(test).select("label", "prediction").show()

Коефіцієнти: [0.6575001543759155,1.3150003087520437]
Intercept:   -0.1867
RMSE: 0.1134
R²:   0.9989
MAE:  0.0933
+-----+------------------+
|label|        prediction|
+-----+------------------+
|  3.8|3.7583328187467706|
|  6.3| 6.388333436250646|
+-----+------------------+



### Кластеризація

Алгоритми кластеризації – задачі **навчання без учителя**: мітки класів не потрібні. Мета – знайти природні групи у даних.

---

#### K-Means

Найпопулярніший алгоритм кластеризації, що розділяє точки на $k$ кластерів.

- **Ініціалізація:** Використовує `k-means||` – паралельну версію ініціалізації `k-means++`, запропоновану Bahmani et al. Це забезпечує якісну початкову розстановку центроїдів навіть на великих розподілених даних.

- **Мета:** Мінімізація суми квадратів відстаней між точками та центроїдами відповідних кластерів.

- **Відстані:** Підтримує евклідову відстань (`euclidean`) та косинусну схожість (`cosine`).

#### BisectingKMeans

Різновид ієрархічної кластеризації з підходом "зверху вниз" (divisive).
- **Механіка:** Починає з одного кластера, який ітеративно розбивається на менші за допомогою `k-means`.

- **Ефективність:** Часто працює швидше за звичайний `k-means`, якщо кількість кластерів велика, і створює деревоподібну структуру.

---

#### Gaussian Mixture Model (GMM)

Ймовірнісна модель, яка припускає, що дані згенеровані із суміші кількох **гаусових (нормальних) розподілів** з невідомими параметрами.

- **Soft Clustering:** На відміну від `k-means`, де точка належить до одного кластера, GMM обчислює **ймовірність** приналежності точки до кожного кластера.

- **Алгоритм:** Використовує ітеративний метод **Expectation-Maximization (EM)** для пошуку параметрів розподілів (середнє значення та коваріація).

#### LDA (Latent Dirichlet Allocation)

Алгоритм тематичного моделювання (topic modeling), який автоматично виявляє приховані теми у колекції документів.

- **Об'єкт:** Сприймає кожен документ як суміш тем, а кожну тему – як розподіл ймовірностей слів.

- **Застосування:** Використовується для кластеризації текстових даних та виявлення прихованих тематичних структур у великих корпусах документів.

---

**Коли що обирати:**

| Алгоритм | Коли використовувати |
|----------|----------------------|
| `KMeans` | Загальний випадок, відома кількість кластерів `k` |
| `BisectingKMeans` | Велике `k`, потрібна швидкість |
| `GaussianMixture` | Кластери різного розміру та форми, потрібні ймовірності |
| `LDA` | Тематичне моделювання тексту |

> [Офіційна документація: Clustering](https://spark.apache.org/docs/latest/ml-clustering.html)

In [ ]:
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.linalg import Vectors

# Два явні кластери у даних
data = spark.createDataFrame([
    (Vectors.dense([0.1, 0.1]),),
    (Vectors.dense([0.0, 0.2]),),
    (Vectors.dense([0.2, 0.0]),),
    (Vectors.dense([9.0, 9.0]),),
    (Vectors.dense([9.1, 8.9]),),
    (Vectors.dense([8.9, 9.1]),),
], ["features"])

# K-Means: k-means|| ініціалізація
kmeans = KMeans(k=2, seed=42)
km_model = kmeans.fit(data)

predictions = km_model.transform(data)
predictions.select("features", "prediction").show()

# Центри кластерів
print("Центри кластерів:")
for center in km_model.clusterCenters():
    print(f"  {center}")

# WSSSE – чим менше, тим кращі кластери (залежить від масштабу даних)
print(f"WSSSE: {km_model.summary.trainingCost:.4f}")

# Silhouette score – від -1 до 1, ближче до 1 = кращі кластери
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette:.4f}")

+---------+----------+
| features|prediction|
+---------+----------+
|[0.1,0.1]|         0|
|[0.0,0.2]|         0|
|[0.2,0.0]|         0|
|[9.0,9.0]|         1|
|[9.1,8.9]|         1|
|[8.9,9.1]|         1|
+---------+----------+

Центри кластерів:
  [0.1 0.1]
  [9. 9.]
WSSSE: 0.0800
Silhouette score: 0.9997


## 7. ML Persistence: Збереження та завантаження PipelineModel

Після навчання `PipelineModel` її можна зберегти на диск і завантажити пізніше — в іншому процесі, на іншому кластері або навіть в іншій мові програмування (Scala/Java/Python). Це є основою для переносу моделей у **Production**.

### Формат збереження на диску

Spark ML зберігає модель у вигляді **директорії** зі строгою ієрархією:
```
my_pipeline_model/
├── metadata/
│   └── part-00000            ← JSON: "паспорт" (PipelineModel, версія Spark, список етапів)
└── stages/
    ├── 0_stringindexer_uid/
    │   ├── metadata/         ← JSON: назви стовпців, параметри
    │   └── data/             ← Parquet: навчений словник (напр. "male" → 0)
    ├── 1_onehotencoder_uid/
    │   ├── metadata/         ← JSON: параметри етапу (dropLast, handleInvalid)
    │   └── data/             ← Parquet: дані про розмірність (cardinality) векторів
    ├── 2_vectorassembler_uid/
    │   └── metadata/         ← JSON: тільки список вхідних стовпців (дані відсутні)
    └── 3_randomforest_uid/
        ├── metadata/         ← JSON: конфігурація лісу (numTrees, maxDepth)
        ├── data/             ← Parquet: база вузлів (умови розщеплення, пороги)
        └── treesMetadata/    ← Parquet: "реєстр" лісу (ваги та ID окремих дерев)
```

**Ключові компоненти:**
- `metadata` / **(JSON):** "Паспорт" етапу або всього пайплайну. Містить назву класу, версію Spark та параметри. Є у кожного етапу без винятку.

- `data` / **(Parquet):** "Навчений стан". Містить знання, отримані під час `.fit()` (словники `StringIndexer`, ваги регресії, вузли дерев). Відсутній у простих Transformers (як-от `VectorAssembler`).

- `treesMetadata` / **(Parquet):** Специфічна папка для **ансамблів** (Random Forest, GBT). Містить реєстр усіх дерев лісу, що дозволяє Spark швидше збирати "голоси" під час прогнозування.

- **Локація:** Підтримуються як локальні шляхи (`/tmp/model`), так і розподілені сховища (`s3://...`, `hdfs://...`, `azure://...`).

### API збереження та завантаження

Усі стандартні компоненти Spark ML, що підтримують збереження, реалізують інтерфейси `MLWritable` та `MLReadable`.

| Метод | Опис |
|-------|------|
| `model.save(path)` | Зберегти об'єкт (модель або Pipeline) у вказаний шлях |
| `model.write().overwrite().save(path)` | Зберегти з примусовим перезаписом, якщо директорія вже існує |
|  `[ClassName].load(path)` | Завантажити об'єкт (напр., `PipelineModel.load(path)` або `KMeans.load(path)`) |

**Крос-мовна сумісність:** Моделі, збережені у Scala або Java, можна завантажити в Python (PySpark) і навпаки. Це можливо завдяки тому, що метадані зберігаються у форматі JSON, а дані моделі – у форматі Parquet.

**Сумісність між версіями Spark**
Офіційна політика Spark щодо збережених моделей:

1. **Minor та patch версії (напр., 3.3.x → 3.4.x):**

    - **Зворотна сумісність:** Гарантована. Модель, створена у версії 3.3, має працювати у 3.4.

    - **Пряма сумісність:** Не гарантується (модель із 3.4 може не відкритися у 3.3, якщо додано нові параметри).

2. **Major версії (напр., 2.x → 3.x або 3.x → 4.x):**

    - **Зворотна сумісність:** Не гарантується, але зазвичай підтримується за принципом **best-effort** (максимальні зусилля розробників). При переході на нову Major-версію рекомендується перенавчати та зберігати моделі заново.


> [Офіційна документація: ML persistence: Saving and Loading Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html#ml-persistence-saving-and-loading-pipelines)

### Версіонування та життєвий цикл моделей

Для надійної роботи в Production важливо розрізняти експериментальні та робочі версії моделей.

#### Файлова структура (Simple Practice)

Простий підхід для невеликих проєктів – іменування за схемою `назва_версія_дата`:

```
models/
├── churn_model_v1_2024-01-15/  ← перша ітерація
├── churn_model_v2_2024-02-20/  ← покращена модель
└── churn_model_prod/           ← копія або посилання на актуальну модель
```
#### MLflow (Professional Tracking)

Для професійного трекінгу Spark ML інтегрується з **MLflow**. Це дозволяє не просто зберігати файли, а бачити, з якими параметрами та метриками була навчена кожна модель.

**Логування:** `mlflow.spark.log_model(spark_model, "model")` – зберігає модель разом із середовищем (Conda/Pip), щоб вона запустилася будь-де.

**Завантаження:** `mlflow.spark.load_model("models:/churn/Production")` – завантажує модель не за шляхом на диску, а за її статусом (Stage, Production, Archived).

**Автоматизація:** `mlflow.pyspark.ml.autolog()` – автоматично записує всі параметри (`regParam`, `elasticNetParam`) та метрики при виклику `.fit()`.

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# Будуємо і навчаємо простий Pipeline
df = spark.createDataFrame([
    (0, "cat", 1.5, 1.0),
    (1, "dog", 2.0, 0.0),
    (2, "cat", 1.2, 1.0),
    (3, "dog", 3.1, 0.0),
], ["id", "animal", "value", "label"])

indexer   = StringIndexer(inputCol="animal", outputCol="animalIdx")
assembler = VectorAssembler(inputCols=["animalIdx", "value"],
                             outputCol="features")
rf        = RandomForestClassifier(numTrees=10, seed=42)

pipeline  = Pipeline(stages=[indexer, assembler, rf])
model     = pipeline.fit(df)

# --- Збереження ---
model_path = "/tmp/my_pipeline_model"
model.write().overwrite().save(model_path)
print(f"Модель збережена у: {model_path}")

# --- Завантаження ---
# PipelineModel.load() повертає той самий тип – Transformer
loaded_model = PipelineModel.load(model_path)
print(f"Кількість етапів: {len(loaded_model.stages)}")
print(f"Типи етапів: {[type(s).__name__ for s in loaded_model.stages]}")

# --- Прогноз на нових даних ---
new_data = spark.createDataFrame([
    (10, "cat", 1.8),
    (11, "dog", 2.5),
], ["id", "animal", "value"])

loaded_model.transform(new_data) \
    .select("id", "animal", "value", "prediction") \
    .show()

Модель збережена у: /tmp/my_pipeline_model
Кількість етапів: 3
Типи етапів: ['StringIndexerModel', 'VectorAssembler', 'RandomForestClassificationModel']
+---+------+-----+----------+
| id|animal|value|prediction|
+---+------+-----+----------+
| 10|   cat|  1.8|       1.0|
| 11|   dog|  2.5|       0.0|
+---+------+-----+----------+



## 8. Практичний end-to-end приклад для табличних даних

Зберемо разом усе, що розглянули у цій темі: завантаження даних → Feature Engineering Pipeline → навчання моделі → збереження → прогноз на нових даних.

**Датасет:** Titanic (класична задача класифікації)  
**Задача:** передбачити чи виживе пасажир (`Survived`: 0 або 1)  
**Алгоритм:** `RandomForestClassifier`

**Ознаки які будемо використовувати:**

| Ознака | Тип | Опис |
|--------|-----|------|
| `Pclass` | Числова | Клас квитка (1, 2, 3) |
| `Sex` | Категоріальна | Стать |
| `Age` | Числова | Вік (є пропущені значення) |
| `SibSp` | Числова | Кількість братів/сестер/подружжя на борту |
| `Parch` | Числова | Кількість батьків/дітей на борту |
| `Fare` | Числова | Вартість квитка |
| `Embarked` | Категоріальна | Порт посадки (S, C, Q) |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Завантажуємо датасет Titanic
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

import pandas as pd
pandas_df = pd.read_csv(url)
df = spark.createDataFrame(pandas_df)

# Залишаємо лише потрібні стовпці
cols_to_use = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df.select(cols_to_use)

# Приводимо типи до числових де потрібно
df = df.withColumn("Survived", F.col("Survived").cast(DoubleType())) \
       .withColumn("Pclass",   F.col("Pclass").cast(DoubleType())) \
       .withColumn("Age",      F.col("Age").cast(DoubleType())) \
       .withColumn("Fare",     F.col("Fare").cast(DoubleType()))

print(f"Рядків: {df.count()}, Стовпців: {len(df.columns)}")
df.printSchema()
df.show(5)

# Розділяємо стовпці за типом
numeric_cols = [f.name for f in df.schema.fields
                if f.dataType.typeName() not in ("string")]
string_cols  = [f.name for f in df.schema.fields
                if f.dataType.typeName() == "string"]

# Для числових: null АБО NaN
# Для рядкових: лише null (isnan не застосовна до string)
null_counts = df.select(
    [F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)).alias(c)
     for c in numeric_cols] +
    [F.count(F.when(F.col(c).isNull(), c)).alias(c)
     for c in string_cols]
)

null_counts.show()

Рядків: 891, Стовпців: 8
root
 |-- Survived: double (nullable = true)
 |-- Pclass: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: long (nullable = true)
 |-- Parch: long (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Embarked: string (nullable = true)

+--------+------+------+----+-----+-----+-------+--------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|
+--------+------+------+----+-----+-----+-------+--------+
|     0.0|   3.0|  male|22.0|    1|    0|   7.25|       S|
|     1.0|   1.0|female|38.0|    1|    0|71.2833|       C|
|     1.0|   3.0|female|26.0|    0|    0|  7.925|       S|
|     1.0|   1.0|female|35.0|    1|    0|   53.1|       S|
|     0.0|   3.0|  male|35.0|    0|    0|   8.05|       S|
+--------+------+------+----+-----+-----+-------+--------+
only showing top 5 rows
+--------+------+---+-----+-----+----+---+--------+
|Survived|Pclass|Age|SibSp|Parch|Fare|Sex|Embarked|
+--------+------

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder,
    VectorAssembler, StandardScaler, Imputer
)

# --- Категоріальні ознаки: Sex, Embarked ---
# StringIndexer для обох стовпців одночасно
str_indexer = StringIndexer(
    inputCols=["Sex", "Embarked"],
    outputCols=["SexIdx", "EmbarkedIdx"],
    handleInvalid="keep"   # пропущені Embarked → окремий бакет
)

# OneHotEncoder
ohe = OneHotEncoder(
    inputCols=["SexIdx", "EmbarkedIdx"],
    outputCols=["SexVec", "EmbarkedVec"]
)

# --- Числові ознаки: заповнення пропущених значень ---
# Age має пропущені значення → заповнюємо медіаною
imputer = Imputer(
    inputCols=["Age", "Fare"],
    outputCols=["AgeImp", "FareImp"],
    strategy="median"
)

# --- VectorAssembler: збираємо всі ознаки у вектор ---
assembler = VectorAssembler(
    inputCols=["Pclass", "SexVec", "AgeImp", "SibSp",
               "Parch", "FareImp", "EmbarkedVec"],
    outputCol="raw_features"
)

# --- StandardScaler: стандартизуємо ---
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False   # False – зберігає Sparse сумісність
)

print("Feature Engineering Pipeline визначено: 5 кроків")
print("StringIndexer → OneHotEncoder → Imputer → VectorAssembler → StandardScaler")

Feature Engineering Pipeline визначено: 5 кроків
StringIndexer → OneHotEncoder → Imputer → VectorAssembler → StandardScaler


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Перейменовуємо Survived → label (контракт Spark ML)
df_ml = df.withColumnRenamed("Survived", "label")

# Розбиваємо на тренувальну і тестову вибірки
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()} рядків, Test: {test.count()} рядків")

# Алгоритм
rf = RandomForestClassifier(
    numTrees=50,
    maxDepth=5,
    seed=42,
    labelCol="label",
    featuresCol="features"
)

# Збираємо повний Pipeline: Feature Engineering + модель
pipeline = Pipeline(stages=[
    str_indexer, ohe, imputer, assembler, scaler, rf
])

# Навчання – fit() виконує всі кроки послідовно
model = pipeline.fit(train)
print("Pipeline навчено успішно")

# Важливість ознак з Random Forest
rf_model = model.stages[-1]  # останній етап – навчена модель

# Отримуємо назви ознак після VectorAssembler
attrs = model.stages[3].getInputCols() # шлях до імен залежить від індексу асемблера
importance_list = sorted(zip(attrs, rf_model.featureImportances), key=lambda x: x[1], reverse=True)

for name, imp in importance_list:
    print(f"{name}: {imp:.4f}")

Train: 734 рядків, Test: 157 рядків
Pipeline навчено успішно
SexVec: 0.2951
AgeImp: 0.2661
Pclass: 0.1241
EmbarkedVec: 0.1206
SibSp: 0.0838
Parch: 0.0480
FareImp: 0.0289


In [ ]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# --- Збереження ---
model_path = "/tmp/titanic_pipeline_model"
model.write().overwrite().save(model_path)
print(f"Модель збережена: {model_path}")

# --- Завантаження ---
loaded_model = PipelineModel.load(model_path)
print(f"Модель завантажена. Кількість етапів: {len(loaded_model.stages)}")

# --- Прогноз на тестових даних ---
predictions = loaded_model.transform(test)
predictions.select("label", "prediction", "probability").show(10)

# --- Базова метрика (детально оцінки розглянемо згодом) ---
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"\nAccuracy на тестових даних: {accuracy:.4f}")

Модель збережена: /tmp/titanic_pipeline_model
Модель завантажена. Кількість етапів: 6
+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       0.0|[0.62017196163128...|
|  0.0|       0.0|[0.62020989066121...|
|  0.0|       0.0|[0.68124497081316...|
|  0.0|       0.0|[0.64325060960742...|
|  0.0|       0.0|[0.72825065706377...|
|  0.0|       0.0|[0.73245051752126...|
|  0.0|       0.0|[0.75540023363378...|
|  0.0|       0.0|[0.75435243604119...|
|  0.0|       0.0|[0.85800336192442...|
|  0.0|       0.0|[0.85739150342915...|
+-----+----------+--------------------+
only showing top 10 rows

Accuracy на тестових даних: 0.8025


## 9. Корисні ресурси

1. [Spark SQL, DataFrames and Datasets Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)

2. [PySpark ML API Reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html)

3. [Spark MLlib Programming Guide](https://spark.apache.org/docs/latest/ml-guide.html)

4. [MLflow & Spark Integration](https://mlflow.org/docs/latest/ml/traditional-ml/sparkml/)